In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────
!pip install transformers datasets torch -q

In [ ]:
# ── CELL 2: Imports ─────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# ── CELL 3 (FINAL FIX) ──────────────────────────────────────
import random, json

# Install huggingface_hub to download Xet-stored files properly
# !pip install huggingface_hub -q   ← run this first if not already installed

from huggingface_hub import hf_hub_download

filepath = hf_hub_download(
    repo_id="Hello-SimpleAI/HC3",
    filename="all.jsonl",
    repo_type="dataset"
)

print(f"Downloaded to: {filepath}")

ai_texts, human_texts = [], []

with open(filepath, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        for ans in item.get("chatgpt_answers", []):
            if ans and len(ans.strip()) > 50:
                ai_texts.append(ans.strip())
        for ans in item.get("human_answers", []):
            if ans and len(ans.strip()) > 50:
                human_texts.append(ans.strip())

random.seed(42)
ai_sample    = random.sample(ai_texts,    min(1000, len(ai_texts)))
human_sample = random.sample(human_texts, min(1000, len(human_texts)))

print(f"Total AI texts found:    {len(ai_texts)}")
print(f"Total Human texts found: {len(human_texts)}")
print(f"\nSampled AI:    {len(ai_sample)}")
print(f"Sampled Human: {len(human_sample)}")
print(f"\nExample AI text:\n{ai_sample[0][:200]}")
print(f"\nExample Human text:\n{human_sample[0][:200]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


all.jsonl:   0%|          | 0.00/73.7M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--Hello-SimpleAI--HC3/snapshots/4d0ff18143b5a7e1b1e79beb540c04549d1e59d3/all.jsonl
Total AI texts found:    26839
Total Human texts found: 57552

Sampled AI:    1000
Sampled Human: 1000

Example AI text:
Peter J. Denning is a computer scientist and professor emeritus at the United States Naval Academy. He is known for his research and writing in the fields of operating systems, computer networks, and 

Example Human text:
Exit polling is when survey takers stand at the voting / polling location exits ( hence the name ) and take surveys of a random sample of people leaving the voting location . They usually choose preci


In [ ]:
# ── CELL 4: Load RoBERTa detector ───────────────────────────
# This model was fine-tuned specifically to detect ChatGPT text
# Label 0 = Human,  Label 1 = ChatGPT/AI

MODEL_NAME = "Hello-SimpleAI/chatgpt-detector-roberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model     = model.to(device)
model.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"Labels: {model.config.id2label}")

config.json:   0%|          | 0.00/858 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/391 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: Hello-SimpleAI/chatgpt-detector-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: Hello-SimpleAI/chatgpt-detector-roberta
Labels: {0: 'Human', 1: 'ChatGPT'}


In [ ]:
# ── CELL 5: Inference function ───────────────────────────────
def get_predictions(texts, batch_size=16, max_length=512):
    """
    Returns:
        preds      : list of predicted labels (0=human, 1=AI)
        ai_probs   : list of AI-class probabilities
    """
    preds    = []
    ai_probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Running inference"):
        batch = texts[i : i + batch_size]

        # Tokenize — truncate long texts to 512 tokens
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            probs   = torch.softmax(outputs.logits, dim=-1)  # [batch, 2]

        # Class 1 = AI/ChatGPT
        ai_prob  = probs[:, 1].cpu().numpy()
        pred     = np.argmax(probs.cpu().numpy(), axis=-1)

        preds.extend(pred.tolist())
        ai_probs.extend(ai_prob.tolist())

    return preds, ai_probs

In [ ]:
# ── CELL 10: RoBERTa at 1000 samples ────────────────────────
# (Reuses get_predictions() from Cell 5 — run that first)

print("Running RoBERTa on 1000 samples...")

ai_preds_1k,    ai_conf_1k    = get_predictions(ai_sample)
human_preds_1k, human_conf_1k = get_predictions(human_sample)

roberta_ai_dr  = sum(p == 1 for p in ai_preds_1k)    / len(ai_preds_1k)    * 100
roberta_fpr    = sum(p == 1 for p in human_preds_1k)  / len(human_preds_1k) * 100
roberta_ai_avg = float(np.mean(ai_conf_1k))    * 100
roberta_hu_avg = float(np.mean(human_conf_1k)) * 100

print(f"RoBERTa  |  AI Detection Rate: {roberta_ai_dr:.1f}%  |  FPR: {roberta_fpr:.1f}%  |  Avg AI conf: {roberta_ai_avg:.1f}%")

Running RoBERTa on 1000 samples...


Running inference: 100%|██████████| 63/63 [00:22<00:00,  2.74it/s]

RoBERTa  |  AI Detection Rate: 98.9%  |  FPR: 0.5%  |  Avg AI conf: 98.9%


In [ ]:
del model
torch.cuda.empty_cache()
print(f"VRAM freed: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB available")

VRAM freed: 15.5 GB available


In [ ]:
# ── CELL 11: DetectGPT ───────────────────────────────────────
# DetectGPT scores text by:
# 1. Making N random perturbations of the text (using a mask-fill model)
# 2. Checking if the original scores higher log-prob than perturbations
# AI text tends to sit at probability peaks → original >> perturbations
# Human text is more random → original ≈ perturbations
#
# We use a lightweight implementation with GPT-2 as the scoring model
# and T5 as the mask-filling model. Slow but no extra GPU memory needed.

# !pip install transformers -q  # already installed

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm import tqdm

# --- Load scoring model (GPT-2) ---
print("Loading GPT-2 for scoring...")
score_tokenizer = AutoTokenizer.from_pretrained("gpt2")
score_model     = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
score_model.eval()

# --- Load mask-fill model (T5 small for speed) ---
print("Loading T5 for perturbations...")
from transformers import T5ForConditionalGeneration, T5Tokenizer

t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model     = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
t5_model.eval()
print("T5 loaded.")

def get_log_prob(text, tokenizer, model, max_len=256):
    """Compute average token log-probability of text under the model."""
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=max_len
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return -outputs.loss.item()   # loss = NLL, so negate for log-prob


def perturb_text(text, n_perturbations=20, mask_prob=0.15):
    words = text.split()
    if len(words) < 5:
        return [text] * n_perturbations

    perturbations = []
    for _ in range(n_perturbations):
        masked = words.copy()
        indices = random.sample(range(len(masked)), max(1, int(len(masked) * mask_prob)))
        for idx in indices:
            masked[idx] = "<extra_id_0>"
        masked_text = " ".join(masked)

        try:
            inputs = t5_tokenizer(
                masked_text, return_tensors="pt",
                truncation=True, max_length=256
            ).to(device)
            with torch.no_grad():
                outputs = t5_model.generate(
                    **inputs, max_new_tokens=20, do_sample=True, temperature=0.8
                )
            filled = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
            fill_word = filled.strip().split()[0] if filled.strip() else words[indices[0]]
            result = [w if w != "<extra_id_0>" else fill_word for w in masked]
            perturbations.append(" ".join(result))
        except Exception:
            perturbations.append(text)

    return perturbations


def detectgpt_score(text, n_perturbations=20):
    """
    Returns DetectGPT score: original_log_prob - mean(perturbed_log_probs)
    Positive = likely AI (original sits at a peak)
    Negative = likely human (original is not a local peak)
    """
    orig_lp    = get_log_prob(text, score_tokenizer, score_model)
    perturbs   = perturb_text(text, n_perturbations=n_perturbations)
    perturb_lps = [get_log_prob(p, score_tokenizer, score_model) for p in perturbs]
    return orig_lp - float(np.mean(perturb_lps))


# Run on a subset — DetectGPT is slow (each sample takes ~5-10s)
# 200 samples takes ~20-30 min on T4, which is manageable
DETECTGPT_N = 200   # increase to 500 if you have time / Colab Pro

print(f"\nRunning DetectGPT on {DETECTGPT_N} AI + {DETECTGPT_N} Human samples...")
print("Expected time: ~20-40 min on T4 GPU\n")

ai_dgpt_scores    = []
human_dgpt_scores = []

for text in tqdm(ai_sample[:DETECTGPT_N], desc="DetectGPT on AI"):
    ai_dgpt_scores.append(detectgpt_score(text))

for text in tqdm(human_sample[:DETECTGPT_N], desc="DetectGPT on Human"):
    human_dgpt_scores.append(detectgpt_score(text))

# Threshold at 0: score > 0 = AI, score <= 0 = human
THRESHOLD = 0.0
dgpt_ai_detected  = sum(s > THRESHOLD for s in ai_dgpt_scores)
dgpt_human_flagged = sum(s > THRESHOLD for s in human_dgpt_scores)

dgpt_ai_dr  = dgpt_ai_detected   / DETECTGPT_N * 100
dgpt_fpr    = dgpt_human_flagged  / DETECTGPT_N * 100
dgpt_ai_avg = float(np.mean(ai_dgpt_scores))
dgpt_hu_avg = float(np.mean(human_dgpt_scores))

print(f"\nDetectGPT  |  AI Detection Rate: {dgpt_ai_dr:.1f}%  |  FPR: {dgpt_fpr:.1f}%")
print(f"           |  Avg AI score: {dgpt_ai_avg:.4f}  |  Avg Human score: {dgpt_hu_avg:.4f}")

Loading GPT-2 for scoring...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading T5 for perturbations...


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5 loaded.

Running DetectGPT on 200 AI + 200 Human samples...
Expected time: ~20-40 min on T4 GPU



DetectGPT on Human: 100%|██████████| 200/200 [14:35<00:00,  4.38s/it]


DetectGPT  |  AI Detection Rate: 100.0%  |  FPR: 99.5%
           |  Avg AI score: 1.2729  |  Avg Human score: 0.8043


In [ ]:
# Cell 11 - updated cleanup
del t5_model, t5_tokenizer, score_model
torch.cuda.empty_cache()
print(f"VRAM freed: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB available")

VRAM freed: 15.5 GB available


In [ ]:
# ── CELL 12: Binoculars ─────────────────────────────────────
# Binoculars uses the ratio: perplexity(falcon) / cross-perplexity(falcon, falcon-instruct)
# Low ratio = AI text.  High ratio = human text.
#
# Falcon-7B needs ~14GB VRAM — borderline on T4, safe on A100.
# If you get OOM: switch to falcon-rw-1b (1B params, ~3GB) below.

# !pip install accelerate -q

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

USE_SMALL_MODEL = True   # Set False if you have A100 / Colab Pro

if USE_SMALL_MODEL:
    # 1B model — safe on free T4
    OBSERVER  = "tiiuae/falcon-rw-1b"
    PERFORMER = "tiiuae/falcon-rw-1b"   # same model, different prompt prefix
    print("Using Falcon-RW-1B (lightweight Binoculars approximation)")
else:
    # Full Binoculars — needs A100
    OBSERVER  = "tiiuae/falcon-7b"
    PERFORMER = "tiiuae/falcon-7b-instruct"
    print("Using Falcon-7B (full Binoculars)")

print("Loading observer model...")
obs_tok   = AutoTokenizer.from_pretrained(OBSERVER)
obs_model = AutoModelForCausalLM.from_pretrained(
    OBSERVER, torch_dtype=torch.float16, device_map="auto"
)
obs_model.eval()

if USE_SMALL_MODEL:
    per_tok   = obs_tok
    per_model = obs_model
else:
    print("Loading performer model...")
    per_tok   = AutoTokenizer.from_pretrained(PERFORMER)
    per_model = AutoModelForCausalLM.from_pretrained(
        PERFORMER, torch_dtype=torch.float16, device_map="auto"
    )
    per_model.eval()

def get_ppl(text, tokenizer, model, max_len=512):
    """Perplexity of text under model."""
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=max_len
    ).to(model.device)
    with torch.no_grad():
        loss = model(**inputs, labels=inputs["input_ids"]).loss
    return torch.exp(loss).item()


def get_cross_ppl(text, obs_tokenizer, obs_model, per_tokenizer, per_model, max_len=512):
    """
    Cross-perplexity: score text tokens using observer model logits
    but averaged under performer model's token probabilities.
    Simplified: use observer PPL / performer PPL as proxy.
    """
    ppl_obs = get_ppl(text, obs_tokenizer, obs_model, max_len)
    ppl_per = get_ppl(text, per_tokenizer, per_model, max_len)
    return ppl_obs / (ppl_per + 1e-8)   # Binoculars score — low = AI


BINO_N = 200   # same as DetectGPT for consistency

print(f"\nRunning Binoculars on {BINO_N} AI + {BINO_N} Human samples...")

ai_bino_scores    = []
human_bino_scores = []

for text in tqdm(ai_sample[:BINO_N], desc="Binoculars on AI"):
    try:
        ai_bino_scores.append(get_cross_ppl(text, obs_tok, obs_model, per_tok, per_model))
    except Exception:
        ai_bino_scores.append(float('nan'))

for text in tqdm(human_sample[:BINO_N], desc="Binoculars on Human"):
    try:
        human_bino_scores.append(get_cross_ppl(text, obs_tok, obs_model, per_tok, per_model))
    except Exception:
        human_bino_scores.append(float('nan'))

# Remove NaNs
ai_bino_clean    = [s for s in ai_bino_scores    if not np.isnan(s)]
human_bino_clean = [s for s in human_bino_scores if not np.isnan(s)]

# Threshold: scores below median of combined = AI
all_scores  = ai_bino_clean + human_bino_clean
bino_thresh = float(np.median(all_scores))

bino_ai_detected  = sum(s < bino_thresh for s in ai_bino_clean)
bino_human_flagged = sum(s < bino_thresh for s in human_bino_clean)

bino_ai_dr  = bino_ai_detected   / len(ai_bino_clean)  * 100
bino_fpr    = bino_human_flagged  / len(human_bino_clean) * 100

print(f"\nBinoculars  |  AI Detection Rate: {bino_ai_dr:.1f}%  |  FPR: {bino_fpr:.1f}%")
print(f"            |  Threshold used: {bino_thresh:.4f}")
print(f"            |  Avg AI score: {np.mean(ai_bino_clean):.4f}  |  Avg Human score: {np.mean(human_bino_clean):.4f}")

Using Falcon-RW-1B (lightweight Binoculars approximation)
Loading observer model...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]


Running Binoculars on 200 AI + 200 Human samples...


Binoculars on Human: 100%|██████████| 200/200 [00:16<00:00, 12.44it/s]



Binoculars  |  AI Detection Rate: 92.5%  |  FPR: 7.5%
            |  Threshold used: 1.0000
            |  Avg AI score: 1.0000  |  Avg Human score: 1.0000


In [ ]:
# ── CELL 13: Combined Summary ────────────────────────────────
# Add before the summary print in Cell 13
if 'dgpt_ai_dr' not in dir():
    dgpt_ai_dr = dgpt_fpr = dgpt_ai_avg = float('nan')
if 'bino_ai_dr' not in dir():
    bino_ai_dr = bino_fpr = float('nan')
print("\n" + "=" * 65)
print("EVALUATION 2 — FULL BASELINE SUMMARY")
print("=" * 65)
print(f"{'Detector':<15} {'Samples':<10} {'AI Det. Rate':<15} {'FPR':<10} {'Avg AI Conf'}")
print("-" * 65)
print(f"{'RoBERTa':<15} {len(ai_sample):<10} {roberta_ai_dr:<15.1f} {roberta_fpr:<10.1f} {roberta_ai_avg:.1f}%")
print(f"{'DetectGPT':<15} {DETECTGPT_N:<10} {dgpt_ai_dr:<15.1f} {dgpt_fpr:<10.1f} {dgpt_ai_avg:.4f} (raw score)")
print(f"{'Binoculars':<15} {BINO_N:<10} {bino_ai_dr:<15.1f} {bino_fpr:<10.1f} {np.mean(ai_bino_clean):.4f} (ratio)")
print("=" * 65)

# Save all results
import pandas as pd

eval2_df = pd.DataFrame({
    "text":         ai_sample[:BINO_N] + human_sample[:BINO_N],
    "true_label":   ["AI"] * BINO_N + ["Human"] * BINO_N,
    "roberta_pred": ["AI" if p == 1 else "Human" for p in ai_preds_1k[:BINO_N] + human_preds_1k[:BINO_N]],
    "roberta_prob": list(ai_conf_1k[:BINO_N]) + list(human_conf_1k[:BINO_N]),
    "detectgpt_score": ai_dgpt_scores + human_dgpt_scores,
    "binoculars_score": ai_bino_scores + human_bino_scores,
})

eval2_df.to_csv("evaluation2_results.csv", index=False)
print("\nSaved to evaluation2_results.csv")


EVALUATION 2 — FULL BASELINE SUMMARY
Detector        Samples    AI Det. Rate    FPR        Avg AI Conf
-----------------------------------------------------------------
RoBERTa         1000       98.9            0.5        98.9%
DetectGPT       200        100.0           99.5       1.2729 (raw score)
Binoculars      200        92.5            7.5        1.0000 (ratio)

Saved to evaluation2_results.csv


In [ ]:
# Recalibrate DetectGPT threshold using midpoint of AI and human means
dgpt_thresh = (dgpt_ai_avg + dgpt_hu_avg) / 2
print(f"Recalibrated threshold: {dgpt_thresh:.4f}")

dgpt_ai_detected   = sum(s > dgpt_thresh for s in ai_dgpt_scores)
dgpt_human_flagged = sum(s > dgpt_thresh for s in human_dgpt_scores)

dgpt_ai_dr = dgpt_ai_detected  / len(ai_dgpt_scores) * 100
dgpt_fpr   = dgpt_human_flagged / len(human_dgpt_scores) * 100

print(f"DetectGPT recalibrated  |  AI Detection Rate: {dgpt_ai_dr:.1f}%  |  FPR: {dgpt_fpr:.1f}%")

Recalibrated threshold: 1.0386
DetectGPT recalibrated  |  AI Detection Rate: 87.5%  |  FPR: 10.5%


In [ ]:
# Proper lightweight Binoculars — two different small models
# GPT-2 (124M) as observer, GPT-2-medium (345M) as performer
# Different enough to produce meaningful ratios, both fit on T4

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, numpy as np
from tqdm import tqdm

print("Loading GPT-2 as observer...")
obs_tok   = AutoTokenizer.from_pretrained("gpt2")
obs_model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float16).to(device)
obs_model.eval()

print("Loading GPT-2-medium as performer...")
per_tok   = AutoTokenizer.from_pretrained("gpt2-medium")
per_model = AutoModelForCausalLM.from_pretrained("gpt2-medium", dtype=torch.float16).to(device)
per_model.eval()

def get_ppl(text, tokenizer, model, max_len=256):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len).to(device)
    with torch.no_grad():
        loss = model(**inputs, labels=inputs["input_ids"]).loss
    return torch.exp(loss).item()

def bino_score(text):
    # Low ratio = AI (observer PPL low, meaning text is predictable)
    ppl_obs = get_ppl(text, obs_tok, obs_model)
    ppl_per = get_ppl(text, per_tok, per_model)
    return ppl_obs / (ppl_per + 1e-8)

BINO_N = 200
print(f"\nRunning Binoculars (GPT2 / GPT2-medium) on {BINO_N} samples each...")

ai_bino_scores    = [bino_score(t) for t in tqdm(ai_sample[:BINO_N],    desc="Bino AI")]
human_bino_scores = [bino_score(t) for t in tqdm(human_sample[:BINO_N], desc="Bino Human")]

# Threshold = midpoint between means
bino_thresh = (np.mean(ai_bino_scores) + np.mean(human_bino_scores)) / 2
bino_ai_dr  = sum(s < bino_thresh for s in ai_bino_scores)    / BINO_N * 100
bino_fpr    = sum(s < bino_thresh for s in human_bino_scores)  / BINO_N * 100

print(f"\nBinoculars  |  AI Detection Rate: {bino_ai_dr:.1f}%  |  FPR: {bino_fpr:.1f}%")
print(f"            |  Avg AI score: {np.mean(ai_bino_scores):.4f}  |  Avg Human score: {np.mean(human_bino_scores):.4f}")
print(f"            |  Threshold: {bino_thresh:.4f}")

Loading GPT-2 as observer...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading GPT-2-medium as performer...


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Running Binoculars (GPT2 / GPT2-medium) on 200 samples each...


Bino Human: 100%|██████████| 200/200 [00:06<00:00, 30.77it/s]


Binoculars  |  AI Detection Rate: 46.0%  |  FPR: 61.5%
            |  Avg AI score: 1.3996  |  Avg Human score: 1.3308
            |  Threshold: 1.3652
